<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import os
import sys
from datetime import datetime

# 1. إعداد مسارات المكتبات لتعمل أوفلاين داخل حاوية جوبيتر
offline_packages_path = "/home/jovyan/work/storage/packages"
if os.path.exists(offline_packages_path) and offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# 2. المسارات والإعدادات المتوافقة مع بيئة الدوكر وجوبيتر (Docker & Jupyter Native)
PARQUET_PATH = "/home/jovyan/work/storage/historical_parquet"
DB_URL = "jdbc:postgresql://smarthome-postgres:5432/smarthome_energy"

DB_PROPERTIES = {
    "user": "smarthome_user",
    "password": "smarthome_password",
    "driver": "org.postgresql.Driver"
}

print(f"⏰ [{datetime.now()}] جاري بدء تشغيل الـ Batch Job المطور بالكامل داخل Jupyter Notebook...")

# 3. التحقق الأولي من وجود ملفات الباركيه
if not os.path.exists(PARQUET_PATH) or len(os.listdir(PARQUET_PATH)) == 0:
    print("ℹ️ لا توجد ملفات باركيه في الأرشيف لمعالجتها حالياً. يرجى التأكد من تشغيل سكريبت الاستقبال أولاً.")
else:
    # 4. بناء جلسة سبارك الشاملة وتضمين الـ Driver للبوستغرس تلقائياً
    spark = SparkSession.builder \
        .appName("SmartHome-Historical-Compaction-Jupyter") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
        .getOrCreate()

    # تقليل تفاصيل السجلات المزعجة لراحة القراءة داخل جوبيتر
    spark.sparkContext.setLogLevel("ERROR")

    try:
        print("📂 [Spark] جاري قراءة ملفات الباركيه الشاملة من المستودع...")
        df = spark.read.parquet(PARQUET_PATH)

        if df.count() == 0:
            print("ℹ️ ملفات الباركيه فارغة تماماً.")
        else:
            columns_list = df.columns

            # 🎯 توحيد مسميات الأعمدة وتحديد عمود الطاقة القادم من الـ Simulator
            if 'power_consumption_watts' in columns_list:
                df = df.withColumn('power_watts', F.col('power_consumption_watts'))
            elif 'avg_power_watts' in columns_list and 'power_watts' not in columns_list:
                df = df.withColumn('power_watts', F.col('avg_power_watts'))

            if 'power_watts' not in df.columns:
                print(f"❌ خطأ: لم نجد عمود طاقة معرّف. الأعمدة المتاحة هي: {columns_list}")
            else:
                # تنظيف البيانات وتحويلها إلى أرقام وحذف الـ Nulls
                df = df.withColumn('power_watts', F.col('power_watts').cast(DoubleType()))
                df = df.filter(F.col('power_watts').isNotNull())

                total_rows = df.count()
                if total_rows == 0:
                    print("❌ خطأ: بعد التنظيف، لا توجد قراءات رقمية صالحة!")
                else:
                    print(f"⚙️ [Spark] جاري معالجة {total_rows} سجل تراكمي وحساب التجميع الإحصائي الموزع...")

                    # حساب الطاقة المستهلكة التراكمية (كيلوواط ساعة kWh)
                    df = df.withColumn('kwh_consumed', F.col('power_watts') / 1000.0)

                    # التجميع الكلي الشامل وتحويل الأنواع لتطابق البوستغرس بدقة decimal(10,2)
                    summary_df = df.groupBy('zone', 'device_type').agg(
                        F.round(F.sum('kwh_consumed'), 2).cast("decimal(10,2)").alias('overall_avg_watts'),
                        F.round(F.max('power_watts'), 2).cast("decimal(10,2)").alias('peak_power_watts'),
                        F.count('power_watts').alias('total_records_analyzed')
                    )

                    # إضافة عمود التوقيت الحالي ليكون متوافقاً مع الهيكل التاريخي
                    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    summary_df = summary_df.withColumn('last_updated', F.lit(current_time).cast("timestamp"))

                    print("💾 [Spark JDBC] جاري حقن النتائج الملخصة مباشرة في جدول البوستغرس للـ Dashboard...")
                    
                    # الكتابة المباشرة الموزعة إلى البوستغرس عبر الشبكة الداخلية
                    summary_df.write \
                        .mode("overwrite") \
                        .option("truncate", "true") \
                        .jdbc(url=DB_URL, table="historical_analytics_summary", properties=DB_PROPERTIES)

                    print(f"✅ [نجاح باهر] تم تحديث الجدول historical_analytics_summary بنجاح ساحق!")
                    
                    # عرض عينة من النتائج داخل جوبيتر
                    summary_df.show(5, truncate=False)

    except Exception as e:
        print(f"❌ فشل سكريبت المعالجة الدوري داخل جوبيتر: {e}")
    finally:
        # إغلاق الجلسة دائماً لتحرير موارد الـ JVM والذاكرة
        spark.stop()
        print("🔒 تم إغلاق جلسة Spark بنجاح.")

ModuleNotFoundError: No module named 'pyspark'